In [1]:
#!/usr/bin/env python3
"""
clean_all_sheets.py

Cleans all sheets in an input Excel workbook and saves cleaned sheets
into an output workbook named "temp_<original_sheetname>".

Features:
- Drops entirely-empty or "Unnamed" columns
- Strips whitespace and converts empty strings to NaN
- Attempts to collapse multi-row headers into a single row header (heuristic)
- Normalizes column names to snake_case
- Removes rows with any missing values (dropna(how='any'))
- Removes grouping/segregation rows that match configurable keywords
- Saves cleaned sheets to an Excel file and prints a summary per sheet

Dependencies:
- pandas
- openpyxl (for Excel I/O)

Example:
    python clean_all_sheets.py
"""

from pathlib import Path
import re
import argparse
import pandas as pd

# ---------------------------
# Configurable parameters
# ---------------------------
DEFAULT_SEG_KEYWORDS = [
    r"\bhigh(?:est)?\b", r"\bavg\b", r"\baverage\b", r"\bmedian\b", r"\blow(?:est)?\b",
    r"\bvery\s+high\b", r"\bhigh\s+hdi\b", r"\blow\s+hdi\b", r"\bgroup\b", r"\bsub-?region\b",
    r"\bregion\b", r"\btotal\b", r"\bsubtotal\b", r"\bworld\b", r"\bregional\b",
    r"\bpercentiles?\b", r"\btop\b", r"\bbottom\b"
]

# ---------------------------
# Utilities
# ---------------------------
def normalize_colname(name: object) -> str:
    """
    Convert an arbitrary column header to a safe snake_case string.
    """
    if pd.isna(name):
        name = "unnamed"
    name = str(name).strip()
    if name == "" or re.fullmatch(r"[-\s\.]*", name):
        name = "unnamed"
    # Replace non-word characters with underscore
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_").lower()
    if name == "":
        name = "unnamed"
    return name

def looks_like_all_whitespace_or_na(series: pd.Series) -> bool:
    """
    True if the series has no non-empty values (after stripping).
    """
    # Drop nulls then check remaining values are whitespace/empty
    non_null = series.dropna()
    if non_null.shape[0] == 0:
        return True
    return non_null.apply(lambda x: str(x).strip() == "").all()

def collapse_multirow_header_if_present(df: pd.DataFrame) -> pd.DataFrame:
    """
    Heuristic: if many column names are generic Unnamed and the first row
    contains non-empty strings for many columns, use the first row as header.
    """
    # Count 'Unnamed' style column labels
    num_unnamed = sum(1 for c in df.columns if isinstance(c, str) and re.fullmatch(r"Unnamed.*", str(c)))
    if num_unnamed > len(df.columns) // 2 and len(df) > 0:
        first_row = df.iloc[0].astype(str).fillna("")
        nonempty_ratio = (first_row != "").mean()
        if nonempty_ratio > 0.4:
            df = df.copy()
            df.columns = first_row
            df = df.drop(df.index[0]).reset_index(drop=True)
    return df

# ---------------------------
# Main cleaning function
# ---------------------------
def clean_dataframe(df: pd.DataFrame, seg_pattern: re.Pattern) -> (pd.DataFrame, dict):
    """
    Clean a single DataFrame and return it with a small metadata dict.

    Steps:
    - strip strings, replace whitespace-only with pd.NA
    - drop columns that are entirely empty or labelled Unnamed
    - collapse multi-row header heuristically
    - normalize column names to snake_case and ensure uniqueness
    - drop any row containing any NaN (dropna(how='any'))
    - remove rows that match segmentation/grouping keywords
    """
    meta = {"cols_dropped": [], "rows_before": len(df), "cols_before": len(df.columns)}

    df = df.copy()

    # Strip whitespace from string cells, convert empty strings to pd.NA
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.replace(r'^\s*$', pd.NA, regex=True)

    # Drop columns that are entirely NA / whitespace or have header "Unnamed"
    cols_to_drop = []
    for col in df.columns:
        if isinstance(col, str) and col.strip().lower().startswith("unnamed"):
            cols_to_drop.append(col)
            continue
        series = df[col]
        if looks_like_all_whitespace_or_na(series):
            cols_to_drop.append(col)
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        meta["cols_dropped"] = cols_to_drop.copy()

    # Try collapsing multi-row header (heuristic)
    df = collapse_multirow_header_if_present(df)

    # Normalize column names to snake_case & enforce uniqueness
    new_cols = []
    counts = {}
    for col in df.columns:
        new = normalize_colname(col)
        # enforce uniqueness by suffixing integer if needed
        if new in counts:
            counts[new] += 1
            new = f"{new}_{counts[new]}"
        else:
            counts[new] = 0
        new_cols.append(new)
    df.columns = new_cols

    # Remove rows with any missing values
    df = df.dropna(axis=0, how="any").reset_index(drop=True)

    # Remove segmentation rows: if any string cell matches seg_pattern, drop row
    def row_is_segmentation(row):
        for v in row:
            if pd.isna(v):
                continue
            if isinstance(v, (int, float)):
                continue
            s = str(v).strip()
            if s == "":
                continue
            if seg_pattern.search(s):
                return True
        return False

    mask = df.apply(row_is_segmentation, axis=1)
    if mask.any():
        df = df[~mask].reset_index(drop=True)

    meta["rows_after"] = len(df)
    meta["cols_after"] = len(df.columns)
    return df, meta

# ---------------------------
# Orchestration
# ---------------------------
def process_workbook(input_path: Path, output_path: Path, seg_keywords=None, verbose=True):
    if seg_keywords is None:
        seg_keywords = DEFAULT_SEG_KEYWORDS
    seg_pattern = re.compile("|".join(seg_keywords), flags=re.I)

    # Read sheets
    xls = pd.ExcelFile(input_path)
    cleaned_sheets = {}
    summary = {}

    for sheet in xls.sheet_names:
        # Read sheet as object dtype to preserve strings exactly
        try:
            raw = pd.read_excel(xls, sheet_name=sheet, header=0, dtype=object)
        except Exception:
            raw = pd.read_excel(xls, sheet_name=sheet, header=0)

        cleaned_df, meta = clean_dataframe(raw, seg_pattern)
        cleaned_sheets[sheet] = cleaned_df
        summary[sheet] = meta

        if verbose:
            print(f"Sheet: '{sheet}' | rows {meta['rows_before']} -> {meta['rows_after']} | "
                  f"cols {meta['cols_before']} -> {meta['cols_after']} | dropped cols: {len(meta['cols_dropped'])}")

    # Write cleaned sheets into an Excel workbook with names "temp_<sheet>"
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet, df in cleaned_sheets.items():
            out_name = f"temp_{sheet}"[:31]  # Excel sheet name length limit
            df.to_excel(writer, sheet_name=out_name, index=False)

    if verbose:
        print(f"\nSaved cleaned workbook to: {output_path}")
    return summary

# ---------------------------
# CLI
# ---------------------------
def main():
    parser = argparse.ArgumentParser(description="Clean all sheets in an Excel workbook.")
    parser.add_argument("--input", "-i", type=str, default="/mnt/data/HDR25_Statistical_Annex_Tables_1-7(1).xlsx",
                        help="Path to input Excel file")
    parser.add_argument("--output", "-o", type=str, default="/mnt/data/HDR25_cleaned_temp.xlsx",
                        help="Path for output cleaned Excel file")
    parser.add_argument("--no-verbose", action="store_true", help="Suppress per-sheet printing")
    args = parser.parse_args()

    input_path = Path(args.input)
    output_path = Path(args.output)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    summary = process_workbook(input_path, output_path, verbose=not args.no_verbose)

    # Print a nicer summary table
    df_summary = pd.DataFrame.from_dict(summary, orient="index")
    print("\nSummary per sheet:")
    print(df_summary[["rows_before", "rows_after", "cols_before", "cols_after"]])

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--input INPUT] [--output OUTPUT]
                             [--no-verbose]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\user\AppData\Roaming\jupyter\runtime\kernel-cb924602-a19e-4699-88a0-371f65af147a.json


SystemExit: 2

C:\Users\user\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
#!/usr/bin/env python3
"""
clean_all_sheets.py  -- fixed to work inside Jupyter / normal script

Use as:
  python clean_all_sheets.py --input path/to/file.xlsx --output path/to/out.xlsx

Or, in a Jupyter cell, just run the cell (it will use the defaults or you can call process_workbook manually).
"""
from pathlib import Path
import re
import argparse
import pandas as pd

# ---------------------------
# Configurable parameters
# ---------------------------
DEFAULT_SEG_KEYWORDS = [
    r"\bhigh(?:est)?\b", r"\bavg\b", r"\baverage\b", r"\bmedian\b", r"\blow(?:est)?\b",
    r"\bvery\s+high\b", r"\bhigh\s+hdi\b", r"\blow\s+hdi\b", r"\bgroup\b", r"\bsub-?region\b",
    r"\bregion\b", r"\btotal\b", r"\bsubtotal\b", r"\bworld\b", r"\bregional\b",
    r"\bpercentiles?\b", r"\btop\b", r"\bbottom\b"
]

# ---------------------------
# Utilities
# ---------------------------
def normalize_colname(name: object) -> str:
    if pd.isna(name):
        name = "unnamed"
    name = str(name).strip()
    if name == "" or re.fullmatch(r"[-\s\.]*", name):
        name = "unnamed"
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_").lower()
    if name == "":
        name = "unnamed"
    return name

def looks_like_all_whitespace_or_na(series: pd.Series) -> bool:
    non_null = series.dropna()
    if non_null.shape[0] == 0:
        return True
    return non_null.apply(lambda x: str(x).strip() == "").all()

def collapse_multirow_header_if_present(df: pd.DataFrame) -> pd.DataFrame:
    num_unnamed = sum(1 for c in df.columns if isinstance(c, str) and re.fullmatch(r"Unnamed.*", str(c)))
    if num_unnamed > len(df.columns) // 2 and len(df) > 0:
        first_row = df.iloc[0].astype(str).fillna("")
        nonempty_ratio = (first_row != "").mean()
        if nonempty_ratio > 0.4:
            df = df.copy()
            df.columns = first_row
            df = df.drop(df.index[0]).reset_index(drop=True)
    return df

# ---------------------------
# Main cleaning function
# ---------------------------
def clean_dataframe(df: pd.DataFrame, seg_pattern: re.Pattern) -> (pd.DataFrame, dict):
    meta = {"cols_dropped": [], "rows_before": len(df), "cols_before": len(df.columns)}
    df = df.copy()
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.replace(r'^\s*$', pd.NA, regex=True)

    cols_to_drop = []
    for col in df.columns:
        if isinstance(col, str) and col.strip().lower().startswith("unnamed"):
            cols_to_drop.append(col)
            continue
        series = df[col]
        if looks_like_all_whitespace_or_na(series):
            cols_to_drop.append(col)
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        meta["cols_dropped"] = cols_to_drop.copy()

    df = collapse_multirow_header_if_present(df)

    new_cols = []
    counts = {}
    for col in df.columns:
        new = normalize_colname(col)
        if new in counts:
            counts[new] += 1
            new = f"{new}_{counts[new]}"
        else:
            counts[new] = 0
        new_cols.append(new)
    df.columns = new_cols

    df = df.dropna(axis=0, how="any").reset_index(drop=True)

    def row_is_segmentation(row):
        for v in row:
            if pd.isna(v):
                continue
            if isinstance(v, (int, float)):
                continue
            s = str(v).strip()
            if s == "":
                continue
            if seg_pattern.search(s):
                return True
        return False

    mask = df.apply(row_is_segmentation, axis=1)
    if mask.any():
        df = df[~mask].reset_index(drop=True)

    meta["rows_after"] = len(df)
    meta["cols_after"] = len(df.columns)
    return df, meta

# ---------------------------
# Orchestration
# ---------------------------
def process_workbook(input_path: Path, output_path: Path, seg_keywords=None, verbose=True):
    if seg_keywords is None:
        seg_keywords = DEFAULT_SEG_KEYWORDS
    seg_pattern = re.compile("|".join(seg_keywords), flags=re.I)

    xls = pd.ExcelFile(input_path)
    cleaned_sheets = {}
    summary = {}

    for sheet in xls.sheet_names:
        try:
            raw = pd.read_excel(xls, sheet_name=sheet, header=0, dtype=object)
        except Exception:
            raw = pd.read_excel(xls, sheet_name=sheet, header=0)

        cleaned_df, meta = clean_dataframe(raw, seg_pattern)
        cleaned_sheets[sheet] = cleaned_df
        summary[sheet] = meta

        if verbose:
            print(f"Sheet: '{sheet}' | rows {meta['rows_before']} -> {meta['rows_after']} | "
                  f"cols {meta['cols_before']} -> {meta['cols_after']} | dropped cols: {len(meta['cols_dropped'])}")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet, df in cleaned_sheets.items():
            out_name = f"temp_{sheet}"[:31]
            df.to_excel(writer, sheet_name=out_name, index=False)

    if verbose:
        print(f"\nSaved cleaned workbook to: {output_path}")
    return summary

# ---------------------------
# CLI / Notebook-safe entry
# ---------------------------
def main(argv=None):
    parser = argparse.ArgumentParser(description="Clean all sheets in an Excel workbook.")
    parser.add_argument("--input", "-i", type=str, default="/mnt/data/HDR25_Statistical_Annex_Tables_1-7(1).xlsx",
                        help="Path to input Excel file")
    parser.add_argument("--output", "-o", type=str, default="/mnt/data/HDR25_cleaned_temp.xlsx",
                        help="Path for output cleaned Excel file")
    parser.add_argument("--no-verbose", action="store_true", help="Suppress per-sheet printing")
    # parse_known_args avoids failing on Jupyter's extra args
    args, unknown = parser.parse_known_args(argv)
    input_path = Path(args.input)
    output_path = Path(args.output)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    summary = process_workbook(input_path, output_path, verbose=not args.no_verbose)

    df_summary = pd.DataFrame.from_dict(summary, orient="index")
    print("\nSummary per sheet:")
    print(df_summary[["rows_before", "rows_after", "cols_before", "cols_after"]])

if __name__ == "__main__":
    # If running in a script, sys.argv will be used. In Jupyter, parse_known_args ignores extra kernel args.
    main()


FileNotFoundError: Input file not found: \mnt\data\HDR25_Statistical_Annex_Tables_1-7(1).xlsx

In [11]:
import pandas as pd
import re
from pathlib import Path

# -------------------------------
# Helper: clean column names
# -------------------------------
def clean_colnames(cols):
    new_cols = []
    for c in cols:
        if not isinstance(c, str):
            c = str(c)

        # lowercase
        c = c.lower()

        # remove special chars
        c = re.sub(r"[^a-z0-9]+", "_", c)

        # remove leading/trailing underscores
        c = c.strip("_")

        new_cols.append(c)
    return new_cols


# -------------------------------
# Helper: detect segregation rows
# e.g. "highest hdi", "avg hdi", "eapc", etc.
# -------------------------------
def is_segmentation_row(row):
    row_str = " ".join([str(x).lower() for x in row if pd.notna(x)])
    keywords = [
        "highest hdi", "lowest hdi", "avg hdi", "average hdi",
        "eapc", "eatr", "high human development",
        "medium human development", "low human development"
    ]
    return any(k in row_str for k in keywords)


# -------------------------------
# Main cleaning function for a sheet
# -------------------------------
def clean_sheet(df):
    # drop columns that are fully empty
    df = df.dropna(axis=1, how="all")

    # drop rows that contain segmentation headers
    df = df[~df.apply(lambda row: is_segmentation_row(row), axis=1)]

    # drop rows with ANY missing values
    df = df.dropna(axis=0, how="any")

    # clean column names
    df.columns = clean_colnames(df.columns)

    return df


# -------------------------------
# Process entire workbook
# -------------------------------
input_file = "C:\\Users\\user\\Downloads\\HDR25_Statistical_Annex_Tables_1-7(1).xlsx"
output_file = r"C:\Users\user\Downloads\temp.xlsx"


excel = pd.ExcelFile(input_file)
cleaned_sheets = {}

for sheet in excel.sheet_names:
    df = excel.parse(sheet)
    cleaned_sheets[sheet] = clean_sheet(df)

# save new workbook
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in cleaned_sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("✔ Cleaning completed.")
print("✔ Saved cleaned workbook to:", output_file)


✔ Cleaning completed.
✔ Saved cleaned workbook to: C:\Users\user\Downloads\temp.xlsx


In [15]:
# careful_read_and_clean.py
# - Detects header row safely (scans first N rows)
# - Promotes header, preserves all original rows until header
# - Drops totally-blank decorative columns first
# - Then drops rows with missing values (configurable allowance)
# - Normalizes column names to snake_case
# - Writes cleaned sheets to a new workbook prefixed with temp_
#
# Usage: run in notebook or as script. Update INPUT_PATH / OUTPUT_PATH to your paths.

import pandas as pd
import re
from pathlib import Path

# ---------- CONFIG ----------
INPUT_PATH = Path("C:\\Users\\user\\Downloads\\HDR25_Statistical_Annex_Tables_1-7(1).xlsx")  # <-- change if needed
OUTPUT_PATH = Path(r"C:\Users\user\Downloads\temp.xlsx")                              # <-- change if needed
ALLOW_MISSING_PER_ROW = 0   # 0 => remove rows with any missing; set >0 to allow tolerance
SCAN_HEADER_ROWS = 10      # how many top rows to scan to find the true header
# Optional manual override: if some sheets need a forced header row (0-indexed), set here.
# Example: {"Sheet1": 2, "Table A": 1}
MANUAL_HEADER_ROW = {
    # "Example Sheet": 2
}

# ---------- helpers ----------
def is_blank(x):
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False

def normalize_colname(x):
    if pd.isna(x):
        s = "unnamed"
    else:
        s = str(x).strip()
        if s == "":
            s = "unnamed"
    s = re.sub(r"[^\w]+", "_", s).strip("_").lower()
    if s == "":
        s = "unnamed"
    return s

def column_is_totally_blank(ser):
    non_na = ser.dropna()
    if non_na.empty:
        return True
    # if all non-na values are whitespace
    return all((isinstance(v, str) and v.strip() == "") for v in non_na)

# score a candidate header row: prefer many non-empty strings, many unique values, low numeric proportion
def header_row_score(row):
    vals = list(row)
    n = len(vals)
    nonempty = sum(1 for v in vals if not is_blank(v))
    unique = len({str(v).strip() for v in vals if not is_blank(v)})
    # numeric count (we prefer fewer numeric-only cells in header)
    numeric = 0
    for v in vals:
        if is_blank(v):
            continue
        try:
            float(str(v))
            numeric += 1
        except Exception:
            pass
    # heuristic score
    score = (nonempty / (n+1)) * 2.0 + (unique / (n+1)) * 1.5 - (numeric / (n+1)) * 1.0
    return float(score)

# ---------- core routine for one sheet ----------
def detect_and_promote_header(df_raw, sheet_name, scan_rows=SCAN_HEADER_ROWS):
    """
    Read raw sheet (pandas df with header=None) and detect best header row among first scan_rows.
    Returns (df_promoted, header_row_index, promoted_header_values)
    """
    # If user forced a header row for this sheet, honor it (if valid)
    if sheet_name in MANUAL_HEADER_ROW:
        hr = MANUAL_HEADER_ROW[sheet_name]
        if hr < 0 or hr >= len(df_raw):
            raise ValueError(f"MANUAL_HEADER_ROW for '{sheet_name}' is out of bounds: {hr}")
        header = df_raw.iloc[hr].tolist()
        df_promoted = df_raw.iloc[hr+1:].copy().reset_index(drop=True)
        df_promoted.columns = header
        return df_promoted, hr, header

    max_row_to_check = min(scan_rows, len(df_raw))
    best_score = -1e9
    best_idx = 0
    for i in range(max_row_to_check):
        row = df_raw.iloc[i]
        s = header_row_score(row)
        # small safety: prefer headers that contain at least 2 non-empty cells
        if s > best_score:
            best_score = s
            best_idx = i

    # require a reasonable score threshold to promote; if too low, assume the first row is header (0)
    # threshold is heuristic; adjust if you see wrong behavior.
    header = df_raw.iloc[best_idx].tolist()
    df_promoted = df_raw.iloc[best_idx+1:].copy().reset_index(drop=True)
    df_promoted.columns = header
    return df_promoted, best_idx, header

# ---------- main cleaning over workbook ----------
def clean_workbook(input_path: Path, output_path: Path, allow_missing_per_row=ALLOW_MISSING_PER_ROW):
    if not input_path.exists():
        raise FileNotFoundError(f"Input workbook not found: {input_path}")

    xls = pd.ExcelFile(input_path)
    cleaned_sheets = {}
    diagnostics = []

    for sheet in xls.sheet_names:
        # read raw with header=None so we don't lose the original rows
        df_raw = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=object)
        print(f"\n--- Processing sheet: '{sheet}' (raw rows: {len(df_raw)}, cols: {len(df_raw.columns)}) ---")

        # detect and promote header safely
        df_promoted, header_row_idx, promoted_header = detect_and_promote_header(df_raw, sheet)
        print(f"Detected header row index: {header_row_idx}")
        print("Header preview:", promoted_header[:10])

        # 1) strip whitespace in string cells (makes blank detection reliable)
        df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)

        # 2) drop columns that are entirely blank (decoration)
        cols_before = list(df_promoted.columns)
        blank_cols = [c for c in cols_before if column_is_totally_blank(df_promoted[c])]
        if blank_cols:
            print("Dropping totally-blank decorative columns:", blank_cols)
            df_promoted = df_promoted.drop(columns=blank_cols)
        else:
            print("No totally-blank decorative columns found.")

        # 3) normalize column names to snake_case and ensure uniqueness
        new_cols = []
        counts = {}
        for c in df_promoted.columns:
            n = normalize_colname(c)
            if n in counts:
                counts[n] += 1
                n = f"{n}_{counts[n]}"
            else:
                counts[n] = 0
            new_cols.append(n)
        df_promoted.columns = new_cols

        # 4) drop rows based on missing-value policy
        total_cols = len(df_promoted.columns)
        if total_cols == 0:
            print("Warning: sheet has 0 columns after dropping decorative cols; result will be empty.")
            df_cleaned = df_promoted.copy()
        else:
            if allow_missing_per_row <= 0:
                df_cleaned = df_promoted.dropna(axis=0, how="any").reset_index(drop=True)
            else:
                thresh = total_cols - allow_missing_per_row
                df_cleaned = df_promoted.dropna(axis=0, thresh=thresh).reset_index(drop=True)

        # 5) diagnostics and previews
        diag = {
            "sheet": sheet,
            "raw_rows": len(df_raw),
            "header_row_idx": int(header_row_idx),
            "cols_before": len(cols_before),
            "cols_after_decor_drop": len(df_promoted.columns),
            "rows_after_clean": len(df_cleaned),
            "dropped_blank_columns": blank_cols
        }
        diagnostics.append(diag)

        print(f"Rows before cleaning: {len(df_promoted)} -> after dropping rows: {len(df_cleaned)}")
        if len(df_cleaned) > 0:
            print("Preview (first 6 rows) of cleaned sheet:")
            display(df_cleaned.head(6))
        else:
            print("Resulting cleaned sheet is EMPTY (0 rows). Consider increasing ALLOW_MISSING_PER_ROW.")

        # store cleaned sheet; prefix sheet name with temp_ (Excel limit 31 chars)
        out_name = f"temp_{sheet}"[:31]
        cleaned_sheets[out_name] = df_cleaned

    # write cleaned workbook
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for out_name, df in cleaned_sheets.items():
            df.to_excel(writer, sheet_name=out_name, index=False)

    # show simple diagnostic table
    diag_df = pd.DataFrame(diagnostics).set_index("sheet")
    print("\nSummary diagnostics per sheet:")
    print(diag_df[["raw_rows", "header_row_idx", "cols_before", "cols_after_decor_drop", "rows_after_clean"]])

    return output_path, diag_df

# ---------- execute ----------
if __name__ == "__main__":
    out_file, diag = clean_workbook(INPUT_PATH, OUTPUT_PATH, allow_missing_per_row=ALLOW_MISSING_PER_ROW)
    print("\nSaved cleaned workbook to:", out_file)



--- Processing sheet: 'Table 1. HDI' (raw rows: 279, cols: 15) ---
Detected header row index: 9
Header preview: [2, 'Norway', 0.97, nan, 83.308, nan, 18.79285049, 'c', 13.11796218, 'e']
Dropping totally-blank decorative columns: [nan, nan, nan]
Rows before cleaning: 269 -> after dropping rows: 4
Preview (first 6 rows) of cleaned sheet:


C:\Users\user\AppData\Local\Temp\ipykernel_95864\171525737.py:127: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)


,2,norway,0_97,83_308,18_79285049,c,13_11796218,e,112710_0211,f,0,1
0,4.0,Denmark,0.962,81.933,18.704010,c,13.027321,e,76007.85669,f,4,4
1,11.0,Ireland,0.949,82.412,19.184879,c,11.725,e,90884.63459,f,-6,10
2,121.0,Venezuela (Bolivarian Republic of),0.709,72.514,12.965377,p,9.678385,p,7157.08,u,11,121
3,150.0,Myanmar,0.609,66.889,11.504560,s,6.38,z,4918.810577,aa,3,149



--- Processing sheet: 'Table 2. HDI trends' (raw rows: 242, cols: 27) ---
Detected header row index: 4
Header preview: ['HDI rank', 'Country', 1990, nan, 2000, nan, 2010, nan, 2015, nan]
Dropping totally-blank decorative columns: [nan, nan, nan, nan, nan, nan, nan, nan, 'a', nan, nan, nan]
Rows before cleaning: 237 -> after dropping rows: 193
Preview (first 6 rows) of cleaned sheet:


C:\Users\user\AppData\Local\Temp\ipykernel_95864\171525737.py:127: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)


...

,1,iceland,0_024,7,2_654417805,3_369,47_61904762,99_87591553,99_58273315,70_46,79_31
0,2.0,Norway,0.004,2,1.663741,1.405,46.153846,95.929504,98.505096,62.13,69.2
1,2.0,Switzerland,0.01,4,7.378755,1.483,37.804878,98.03656,98.324165,62.57,72.86
2,4.0,Denmark,0.003,1,4.657717,1.140,43.575419,90.998726,92.48922,59.7,67.7
3,5.0,Germany,0.057,21,4.42844,5.473,35.279503,93.591591,94.346039,56.41,66.72
4,5.0,Sweden,0.007,3,4.507839,1.792,46.418338,94.923492,94.123909,64.38,70.63
5,7.0,Australia,0.056,20,2.941509,6.715,44.493392,92.301262,93.453598,62.81,71.68



--- Processing sheet: 'Table 7. PHDI' (raw rows: 254, cols: 19) ---
Detected header row index: 4
Header preview: [nan, nan, 'Value', nan, 'Value', nan, 'Difference from HDI value (%)', 'a', 'Difference from HDI rank', 'a']
Dropping totally-blank decorative columns: [nan, nan, nan, nan, 'a', 'a', nan, nan, nan, nan]
Rows before cleaning: 249 -> after dropping rows: 211
Preview (first 6 rows) of cleaned sheet:


C:\Users\user\AppData\Local\Temp\ipykernel_95864\171525737.py:127: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)


,value,value_1,difference_from_hdi_value,difference_from_hdi_rank,value_2,tonnes,value_3,tonnes_1,value_4
0,2023,2023,2023,2023,2023,2023,2023,2023,2023
1,0.972,0.735,24.382716,-40,0.756135,10.024832,0.869145,32.215,0.643126
2,0.97,0.723,25.463918,-49,0.745789,7.106036,0.907244,37.5221,0.584335
3,0.97,0.732,24.536082,-41,0.755101,3.721556,0.951422,39.829,0.558779
4,0.962,0.792,17.671518,-6,0.823696,4.612621,0.939791,26.3949,0.707601
5,0.959,0.785,18.1439,-9,0.818521,7.157141,0.906577,24.331,0.730464



Summary diagnostics per sheet:
                     raw_rows  header_row_idx  cols_before  \
sheet                                                        
Table 1. HDI              279               9           15   
Table 2. HDI trends       242               4           27   
Table 3. IHDI             267               4           32   
Table 4. GDI              279               9           26   
Table 5. GII              268               8           20   
Table 7. PHDI             254               4           19   

                     cols_after_decor_drop  rows_after_clean  
sheet                                                         
Table 1. HDI                            12                 4  
Table 2. HDI trends                     15               193  
Table 3. IHDI                           17               193  
Table 4. GDI                            19                 0  
Table 5. GII                            11               192  
Table 7. PHDI                 

In [17]:
# ultra_safe_read_and_prepare_cleaning.py
# Safe workflow: preview -> drop only truly decorative cols -> flag rows with missing values -> allow controlled removal
#
# 1) Set INPUT_PATH to your original workbook (do NOT overwrite it)
# 2) Run the cell, inspect printed diagnostics + previews
# 3) If previews look good, set PERFORM_ROW_REMOVAL=True (default False) or adjust ALLOW_MISSING_PER_ROW
# 4) Re-run the cell to perform removal and write cleaned file

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ---------- CONFIG ----------
INPUT_PATH = Path(r"C:\Users\user\Downloads\HDR25_Statistical_Annex_Tables_1-7(1).xlsx")  # <- original file (no overwrite)
OUTPUT_DIR = Path(r"C:\Users\user\Downloads\cleaned_outputs")  # will be created if missing
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_MISSING_PER_ROW = 0    # 0 = rows must have no missing to be kept; set >0 to allow that many missing cells
PERFORM_ROW_REMOVAL = False  # default False: script WILL NOT remove flagged rows; set True to actually drop rows
MANUAL_HEADER_ROW = {}       # e.g. {"Sheet1": 2}  (0-indexed row to use as header for that sheet)
SCAN_HEADER_ROWS = 8         # number of top rows to consider when auto-detecting header

# ---------- helpers ----------
def is_blank(x):
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False

def column_decorative_score(series: pd.Series) -> float:
    # Fraction of cells that are blank/NA after stripping
    non_na = series.dropna()
    if series.size == 0:
        return 1.0
    blanks = series.apply(lambda v: isinstance(v, str) and v.strip()=="" or pd.isna(v)).sum()
    return blanks / max(1, len(series))

def normalize_colname(x):
    if pd.isna(x):
        s = "unnamed"
    else:
        s = str(x).strip()
        if s == "":
            s = "unnamed"
    s = re.sub(r"[^\w]+", "_", s).strip("_").lower()
    if s == "":
        s = "unnamed"
    return s

def detect_header_row(df_raw):
    # Heuristic: choose row among first SCAN_HEADER_ROWS with many non-blank string values and few numeric-only
    best_i = 0
    best_score = -1e9
    rows_to_check = min(len(df_raw), SCAN_HEADER_ROWS)
    for i in range(rows_to_check):
        row = df_raw.iloc[i].tolist()
        nonblank = sum(0 if is_blank(v) else 1 for v in row)
        unique = len({str(v).strip() for v in row if not is_blank(v)})
        numeric = sum(1 if (not is_blank(v) and _is_numeric_like(v)) else 0 for v in row)
        # score: prefer many nonblank and many unique and few numeric
        score = nonblank * 2.0 + unique * 1.5 - numeric * 1.0
        if score > best_score:
            best_score = score
            best_i = i
    return best_i

def _is_numeric_like(v):
    try:
        float(str(v))
        return True
    except Exception:
        return False

# ---------- main routine ----------
def safe_prepare_and_clean(input_path: Path):
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    xls = pd.ExcelFile(input_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    cleaned_book_name = OUTPUT_DIR / f"cleaned_temp_{timestamp}.xlsx"
    csv_preview_dir = OUTPUT_DIR / f"previews_{timestamp}"
    csv_preview_dir.mkdir(parents=True, exist_ok=True)

    per_sheet_reports = []

    for sheet in xls.sheet_names:
        print("\n" + "="*60)
        print("Sheet:", sheet)
        # read raw with header=None -> preserve everything
        df_raw = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=object)
        print(" raw shape:", df_raw.shape)

        # determine header row (manual override takes precedence)
        if sheet in MANUAL_HEADER_ROW:
            header_idx = MANUAL_HEADER_ROW[sheet]
            if header_idx < 0 or header_idx >= len(df_raw):
                print(f" MANUAL_HEADER_ROW for '{sheet}' out of bounds; ignoring.")
                header_idx = detect_header_row(df_raw)
            else:
                print(f" Using MANUAL header row {header_idx} for sheet {sheet}.")
        else:
            header_idx = detect_header_row(df_raw)
            print(f" Auto-detected header row index: {header_idx}")

        header_preview = df_raw.iloc[header_idx].tolist()
        print(" header preview (first 12 cells):", header_preview[:12])

        # promote header (data below header), but keep df_promoted_full includes earlier rows if you want them
        df_promoted = df_raw.iloc[header_idx+1:].copy().reset_index(drop=True)
        df_promoted.columns = df_raw.iloc[header_idx].tolist()

        # strip whitespace in strings
        df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)

        # conservative decorative-column detection:
        # mark columns as decorative if >= 0.98 fraction blank (very conservative)
        decor_threshold = 0.98
        col_scores = {c: column_decorative_score(df_promoted[c]) for c in df_promoted.columns}
        decorative_cols = [c for c,s in col_scores.items() if s >= decor_threshold]
        print(" decorative cols (very conservative threshold 0.98):", decorative_cols)

        # drop decorative cols
        df_after_cols = df_promoted.drop(columns=decorative_cols) if decorative_cols else df_promoted.copy()

        # normalize column names
        newcols = []
        counts = {}
        for c in df_after_cols.columns:
            n = normalize_colname(c)
            if n in counts:
                counts[n] += 1
                n = f"{n}_{counts[n]}"
            else:
                counts[n] = 0
            newcols.append(n)
        df_after_cols.columns = newcols

        # now identify rows that would be removed under current ALLOW_MISSING_PER_ROW policy
        total_cols = len(df_after_cols.columns)
        if total_cols == 0:
            print(" WARNING: after dropping decorative columns, sheet has 0 columns. Skipping row removal.")
            df_flagged = df_after_cols.copy()
            flagged_mask = pd.Series([False]*len(df_flagged))
        else:
            if ALLOW_MISSING_PER_ROW <= 0:
                # rows that have ANY missing cell
                flagged_mask = df_after_cols.isna().any(axis=1) | df_after_cols.apply(lambda r: any((isinstance(x,str) and x=="") for x in r), axis=1)
            else:
                # compute rows with at least (total_cols - ALLOW_MISSING_PER_ROW) non-NA entries
                required = total_cols - ALLOW_MISSING_PER_ROW
                flagged_mask = df_after_cols.apply(lambda r: r.count() < required, axis=1)

            df_flagged = df_after_cols[flagged_mask]
            print(f" Rows total: {len(df_after_cols)}  Rows flagged for removal (under current policy): {flagged_mask.sum()}")

        # show small previews for manual inspection
        print(" Preview: first 6 rows after dropping decorative columns (not removing flagged rows):")
        display(df_after_cols.head(6))
        if not df_flagged.empty:
            print(" Example flagged rows (first 6) that would be removed if you enable PERFORM_ROW_REMOVAL:")
            display(df_flagged.head(6))

        # Save per-sheet preview CSVs so you can inspect in Excel if you wish
        preview_all_csv = csv_preview_dir / f"{sheet}_after_decor_drop_preview.csv"
        preview_flagged_csv = csv_preview_dir / f"{sheet}_flagged_rows_preview.csv"
        df_after_cols.head(200).to_csv(preview_all_csv, index=False)
        df_flagged.head(200).to_csv(preview_flagged_csv, index=False)

        per_sheet_reports.append({
            "sheet": sheet,
            "raw_rows": len(df_raw),
            "promoted_header_row": int(header_idx),
            "cols_after_header_promotion": len(df_promoted.columns),
            "decorative_cols_dropped": decorative_cols,
            "rows_after_decor_drop": len(df_after_cols),
            "rows_flagged_for_removal": int(flagged_mask.sum()) if total_cols>0 else 0,
            "preview_csv": str(preview_all_csv),
            "flagged_csv": str(preview_flagged_csv)
        })

    # STOP: don't remove flagged rows unless user asked
    if PERFORM_ROW_REMOVAL:
        # Re-run to build final cleaned_sheets dict and write workbook
        cleaned_sheets = {}
        for rpt in per_sheet_reports:
            sheet = rpt["sheet"]
            preview_csv = Path(rpt["preview_csv"])
            # read preview (this is safe copy)
            df_after_cols = pd.read_csv(preview_csv, dtype=object)
            # apply removal according to policy
            if len(df_after_cols.columns)==0:
                df_cleaned = df_after_cols
            else:
                total_cols = len(df_after_cols.columns)
                if ALLOW_MISSING_PER_ROW <= 0:
                    df_cleaned = df_after_cols.dropna(axis=0, how="any").reset_index(drop=True)
                else:
                    thresh = total_cols - ALLOW_MISSING_PER_ROW
                    df_cleaned = df_after_cols.dropna(axis=0, thresh=thresh).reset_index(drop=True)
            cleaned_sheets[f"temp_{sheet}"[:31]] = df_cleaned

        # Save cleaned workbook (safe timestamped name)
        final_name = OUTPUT_DIR / f"cleaned_FINAL_{timestamp}.xlsx"
        from pandas import ExcelWriter
        try:
            with ExcelWriter(final_name, engine="openpyxl") as writer:
                for sname, df in cleaned_sheets.items():
                    df.to_excel(writer, sheet_name=sname, index=False)
            print("\nSaved cleaned workbook with rows removed to:", final_name)
        except PermissionError:
            print("\nPermissionError saving final file. Close target file in Excel and re-run.")
    else:
        print("\nPERFORM_ROW_REMOVAL is False. No rows were removed. To remove flagged rows, set PERFORM_ROW_REMOVAL=True and re-run.")
        print("Preview CSVs (first 200 rows) saved to:", csv_preview_dir)

    # return report df
    return pd.DataFrame(per_sheet_reports).set_index("sheet")

# Run it
report_df = safe_prepare_and_clean(INPUT_PATH)
print("\nSummary report per sheet:")
print(report_df[["raw_rows","promoted_header_row","cols_after_header_promotion","rows_after_decor_drop","rows_flagged_for_removal"]])



Sheet: Table 1. HDI
 raw shape: (279, 15)
 Auto-detected header row index: 4
 header preview (first 12 cells): [nan, nan, 'Human Development Index (HDI) ', nan, 'Life expectancy at birth', nan, 'Expected years of schooling', nan, 'Mean years of schooling', nan, 'Gross national income (GNI) per capita', nan]


C:\Users\user\AppData\Local\Temp\ipykernel_95864\2162972136.py:115: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_promoted = df_promoted.applymap(lambda v: v.strip() if isinstance(v, str) else v)


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [19]:
# fixed_safe_prepare_and_clean.py
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ---------- CONFIG ----------
INPUT_PATH = Path(r"C:\Users\user\Downloads\HDR25_Statistical_Annex_Tables_1-7(1).xlsx")  # <-- update if needed
OUTPUT_DIR = Path(r"C:\Users\user\Downloads\cleaned_outputs")  # preview CSVs and final save go here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_MISSING_PER_ROW = 0    # 0 = rows must have no missing to be kept; set >0 to allow that many missing cells
PERFORM_ROW_REMOVAL = False  # default False: script WILL NOT remove flagged rows; set True to actually drop rows
MANUAL_HEADER_ROW = {}       # e.g. {"Sheet1": 2}  (0-indexed row to use as header for that sheet)
SCAN_HEADER_ROWS = 8         # number of top rows to consider when auto-detecting header

# ---------- helpers ----------
def is_blank(x):
    if pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    return False

def column_decorative_score(series: pd.Series) -> float:
    """
    Fraction of cells considered blank in the series.
    Uses an unambiguous check for blankness (no chained boolean ambiguity).
    """
    total = len(series)
    if total == 0:
        return 1.0
    blanks = 0
    for v in series:
        if pd.isna(v):
            blanks += 1
        elif isinstance(v, str):
            if v.strip() == "":
                blanks += 1
        else:
            # numeric or other non-str non-na -> not blank
            pass
    return blanks / total

def normalize_colname(x):
    if pd.isna(x):
        s = "unnamed"
    else:
        s = str(x).strip()
        if s == "":
            s = "unnamed"
    s = re.sub(r"[^\w]+", "_", s).strip("_").lower()
    if s == "":
        s = "unnamed"
    return s

def detect_header_row(df_raw):
    # Heuristic: choose row among first SCAN_HEADER_ROWS with many non-blank string values and few numeric-only
    best_i = 0
    best_score = -1e9
    rows_to_check = min(len(df_raw), SCAN_HEADER_ROWS)
    for i in range(rows_to_check):
        row = df_raw.iloc[i].tolist()
        nonblank = sum(0 if is_blank(v) else 1 for v in row)
        unique = len({str(v).strip() for v in row if not is_blank(v)})
        numeric = sum(1 if (not is_blank(v) and _is_numeric_like(v)) else 0 for v in row)
        score = nonblank * 2.0 + unique * 1.5 - numeric * 1.0
        if score > best_score:
            best_score = score
            best_i = i
    return best_i

def _is_numeric_like(v):
    try:
        float(str(v))
        return True
    except Exception:
        return False

# ---------- main routine ----------
def safe_prepare_and_clean(input_path: Path):
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    xls = pd.ExcelFile(input_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    cleaned_book_name = OUTPUT_DIR / f"cleaned_temp_{timestamp}.xlsx"
    csv_preview_dir = OUTPUT_DIR / f"previews_{timestamp}"
    csv_preview_dir.mkdir(parents=True, exist_ok=True)

    per_sheet_reports = []

    for sheet in xls.sheet_names:
        print("\n" + "="*70)
        print("Sheet:", sheet)
        df_raw = pd.read_excel(xls, sheet_name=sheet, header=None, dtype=object)
        print(" raw shape:", df_raw.shape)

        # header detection / manual override
        if sheet in MANUAL_HEADER_ROW:
            header_idx = MANUAL_HEADER_ROW[sheet]
            if header_idx < 0 or header_idx >= len(df_raw):
                print(f" MANUAL_HEADER_ROW for '{sheet}' out of bounds; ignoring.")
                header_idx = detect_header_row(df_raw)
            else:
                print(f" Using MANUAL header row {header_idx} for sheet {sheet}.")
        else:
            header_idx = detect_header_row(df_raw)
            print(f" Auto-detected header row index: {header_idx}")

        header_preview = df_raw.iloc[header_idx].tolist()
        print(" header preview (first 12 cells):", header_preview[:12])

        # promote header
        df_promoted = df_raw.iloc[header_idx+1:].copy().reset_index(drop=True)
        df_promoted.columns = df_raw.iloc[header_idx].tolist()

        # strip whitespace in string cells using apply + Series.map (avoids applymap deprecation)
        def strip_series(s: pd.Series) -> pd.Series:
            return s.map(lambda v: v.strip() if isinstance(v, str) else v)
        df_promoted = df_promoted.apply(strip_series, axis=0)

        # decorative-column detection: very conservative threshold
        decor_threshold = 0.98
        col_scores = {c: column_decorative_score(df_promoted[c]) for c in df_promoted.columns}
        decorative_cols = [c for c, s in col_scores.items() if s >= decor_threshold]
        print(" decorative cols (very conservative threshold 0.98):", decorative_cols)

        # drop decorative cols
        df_after_cols = df_promoted.drop(columns=decorative_cols) if decorative_cols else df_promoted.copy()

        # normalize column names
        newcols = []
        counts = {}
        for c in df_after_cols.columns:
            n = normalize_colname(c)
            if n in counts:
                counts[n] += 1
                n = f"{n}_{counts[n]}"
            else:
                counts[n] = 0
            newcols.append(n)
        df_after_cols.columns = newcols

        # identify rows that would be removed
        total_cols = len(df_after_cols.columns)
        if total_cols == 0:
            print(" WARNING: after dropping decorative columns, sheet has 0 columns. Skipping row removal.")
            flagged_mask = pd.Series([False]*len(df_after_cols))
            df_flagged = df_after_cols.copy()
        else:
            if ALLOW_MISSING_PER_ROW <= 0:
                # row is flagged if any cell is NA or empty string
                na_mask = df_after_cols.isna()
                empty_str_mask = df_after_cols.applymap(lambda v: isinstance(v, str) and v == "")
                flagged_mask = na_mask.any(axis=1) | empty_str_mask.any(axis=1)
            else:
                required = total_cols - ALLOW_MISSING_PER_ROW
                flagged_mask = df_after_cols.apply(lambda r: r.count() < required, axis=1)
            df_flagged = df_after_cols[flagged_mask]

            print(f" Rows total: {len(df_after_cols)}  Rows flagged for removal (under current policy): {flagged_mask.sum()}")

        # previews
        print(" Preview: first 6 rows after dropping decorative columns (not removing flagged rows):")
        display(df_after_cols.head(6))
        if not df_flagged.empty:
            print(" Example flagged rows (first 6) that would be removed if you enable PERFORM_ROW_REMOVAL:")
            display(df_flagged.head(6))

        # save per-sheet preview CSVs
        preview_all_csv = csv_preview_dir / f"{sheet}_after_decor_drop_preview.csv"
        preview_flagged_csv = csv_preview_dir / f"{sheet}_flagged_rows_preview.csv"
        df_after_cols.head(200).to_csv(preview_all_csv, index=False)
        df_flagged.head(200).to_csv(preview_flagged_csv, index=False)

        per_sheet_reports.append({
            "sheet": sheet,
            "raw_rows": len(df_raw),
            "promoted_header_row": int(header_idx),
            "cols_after_header_promotion": len(df_promoted.columns),
            "decorative_cols_dropped": decorative_cols,
            "rows_after_decor_drop": len(df_after_cols),
            "rows_flagged_for_removal": int(flagged_mask.sum()) if total_cols>0 else 0,
            "preview_csv": str(preview_all_csv),
            "flagged_csv": str(preview_flagged_csv)
        })

    # If the user requested actual row removal, run it now and save final workbook.
    if PERFORM_ROW_REMOVAL:
        cleaned_sheets = {}
        for rpt in per_sheet_reports:
            sheet = rpt["sheet"]
            preview_csv = Path(rpt["preview_csv"])
            df_after_cols = pd.read_csv(preview_csv, dtype=object)
            if len(df_after_cols.columns) == 0:
                df_cleaned = df_after_cols
            else:
                total_cols = len(df_after_cols.columns)
                if ALLOW_MISSING_PER_ROW <= 0:
                    df_cleaned = df_after_cols.dropna(axis=0, how="any").reset_index(drop=True)
                else:
                    thresh = total_cols - ALLOW_MISSING_PER_ROW
                    df_cleaned = df_after_cols.dropna(axis=0, thresh=thresh).reset_index(drop=True)
            cleaned_sheets[f"temp_{sheet}"[:31]] = df_cleaned

        final_name = OUTPUT_DIR / f"cleaned_FINAL_{timestamp}.xlsx"
        try:
            with pd.ExcelWriter(final_name, engine="openpyxl") as writer:
                for sname, df in cleaned_sheets.items():
                    df.to_excel(writer, sheet_name=sname, index=False)
            print("\nSaved cleaned workbook with rows removed to:", final_name)
        except PermissionError:
            print("\nPermissionError saving final file. Close target file in Excel and re-run.")
    else:
        print("\nPERFORM_ROW_REMOVAL is False. No rows were removed. To remove flagged rows, set PERFORM_ROW_REMOVAL=True and re-run.")
        print("Preview CSVs (first 200 rows) saved to:", csv_preview_dir)

    return pd.DataFrame(per_sheet_reports).set_index("sheet")

# Run it
report_df = safe_prepare_and_clean(INPUT_PATH)
print("\nSummary report per sheet:")
print(report_df[["raw_rows","promoted_header_row","cols_after_header_promotion","rows_after_decor_drop","rows_flagged_for_removal"]])



Sheet: Table 1. HDI
 raw shape: (279, 15)
 Auto-detected header row index: 4
 header preview (first 12 cells): [nan, nan, 'Human Development Index (HDI) ', nan, 'Life expectancy at birth', nan, 'Expected years of schooling', nan, 'Mean years of schooling', nan, 'Gross national income (GNI) per capita', nan]
 decorative cols (very conservative threshold 0.98): []
 Rows total: 274  Rows flagged for removal (under current policy): 274
 Preview: first 6 rows after dropping decorative columns (not removing flagged rows):


C:\Users\user\AppData\Local\Temp\ipykernel_95864\211590439.py:154: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  empty_str_mask = df_after_cols.applymap(lambda v: isinstance(v, str) and v == "")


,unnamed,unnamed_1,human_development_index_hdi,unnamed_2,life_expectancy_at_birth,unnamed_3,expected_years_of_schooling,unnamed_4,mean_years_of_schooling,unnamed_5,gross_national_income_gni_per_capita,unnamed_6,gni_per_capita_rank_minus_hdi_rank,unnamed_7,hdi_rank
0,HDI rank,Country,Value,NaN,(years),NaN,(years),NaN,(years),NaN,(2021 PPP $),NaN,NaN,NaN,NaN
1,NaN,NaN,2023,NaN,2023,NaN,2023,a,2023,a,2023,NaN,2023,b,2022
2,NaN,Very high human development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Iceland,0.972,NaN,82.691,NaN,18.85059,c,13.908926,d,69116.93736,NaN,12,NaN,3
4,2,Norway,0.97,NaN,83.308,NaN,18.79285,c,13.117962,e,112710.0211,f,0,NaN,1
5,2,Switzerland,0.97,NaN,83.954,NaN,16.66753,NaN,13.949121,e,81948.90177,f,5,NaN,2


 Example flagged rows (first 6) that would be removed if you enable PERFORM_ROW_REMOVAL:


,unnamed,unnamed_1,human_development_index_hdi,unnamed_2,life_expectancy_at_birth,unnamed_3,expected_years_of_schooling,unnamed_4,mean_years_of_schooling,unnamed_5,gross_national_income_gni_per_capita,unnamed_6,gni_per_capita_rank_minus_hdi_rank,unnamed_7,hdi_rank
0,HDI rank,Country,Value,NaN,(years),NaN,(years),NaN,(years),NaN,(2021 PPP $),NaN,NaN,NaN,NaN
1,NaN,NaN,2023,NaN,2023,NaN,2023,a,2023,a,2023,NaN,2023,b,2022
2,NaN,Very high human development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Iceland,0.972,NaN,82.691,NaN,18.85059,c,13.908926,d,69116.93736,NaN,12,NaN,3
4,2,Norway,0.97,NaN,83.308,NaN,18.79285,c,13.117962,e,112710.0211,f,0,NaN,1
5,2,Switzerland,0.97,NaN,83.954,NaN,16.66753,NaN,13.949121,e,81948.90177,f,5,NaN,2


...

 raw shape: (254, 19)
 Auto-detected header row index: 4
 header preview (first 12 cells): [nan, nan, 'Value', nan, 'Value', nan, 'Difference from HDI value (%)', 'a', 'Difference from HDI rank', 'a', 'Value', nan]
 decorative cols (very conservative threshold 0.98): []
 Rows total: 249  Rows flagged for removal (under current policy): 249
 Preview: first 6 rows after dropping decorative columns (not removing flagged rows):


,unnamed,unnamed_1,value,unnamed_2,value_1,unnamed_3,difference_from_hdi_value,a,difference_from_hdi_rank,a_1,value_2,unnamed_4,tonnes,unnamed_5,value_3,unnamed_6,tonnes_1,unnamed_7,value_4
0,HDI rank,Country,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023
1,NaN,Very high human development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Iceland,0.972,NaN,0.735,NaN,24.382716,NaN,-40,NaN,0.756135,NaN,10.024832,NaN,0.869145,NaN,32.215,NaN,0.643126
3,2,Norway,0.97,NaN,0.723,NaN,25.463918,NaN,-49,NaN,0.745789,NaN,7.106036,NaN,0.907244,NaN,37.5221,NaN,0.584335
4,2,Switzerland,0.97,NaN,0.732,NaN,24.536082,NaN,-41,NaN,0.755101,NaN,3.721556,NaN,0.951422,NaN,39.829,NaN,0.558779
5,4,Denmark,0.962,NaN,0.792,NaN,17.671518,NaN,-6,NaN,0.823696,NaN,4.612621,NaN,0.939791,NaN,26.3949,NaN,0.707601


 Example flagged rows (first 6) that would be removed if you enable PERFORM_ROW_REMOVAL:


,unnamed,unnamed_1,value,unnamed_2,value_1,unnamed_3,difference_from_hdi_value,a,difference_from_hdi_rank,a_1,value_2,unnamed_4,tonnes,unnamed_5,value_3,unnamed_6,tonnes_1,unnamed_7,value_4
0,HDI rank,Country,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023,NaN,2023
1,NaN,Very high human development,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Iceland,0.972,NaN,0.735,NaN,24.382716,NaN,-40,NaN,0.756135,NaN,10.024832,NaN,0.869145,NaN,32.215,NaN,0.643126
3,2,Norway,0.97,NaN,0.723,NaN,25.463918,NaN,-49,NaN,0.745789,NaN,7.106036,NaN,0.907244,NaN,37.5221,NaN,0.584335
4,2,Switzerland,0.97,NaN,0.732,NaN,24.536082,NaN,-41,NaN,0.755101,NaN,3.721556,NaN,0.951422,NaN,39.829,NaN,0.558779
5,4,Denmark,0.962,NaN,0.792,NaN,17.671518,NaN,-6,NaN,0.823696,NaN,4.612621,NaN,0.939791,NaN,26.3949,NaN,0.707601



PERFORM_ROW_REMOVAL is False. No rows were removed. To remove flagged rows, set PERFORM_ROW_REMOVAL=True and re-run.
Preview CSVs (first 200 rows) saved to: C:\Users\user\Downloads\cleaned_outputs\previews_20251209_130028

Summary report per sheet:
                     raw_rows  promoted_header_row  \
sheet                                                
Table 1. HDI              279                    4   
Table 2. HDI trends       242                    4   
Table 3. IHDI             267                    4   
Table 4. GDI              279                    5   
Table 5. GII              268                    3   
Table 7. PHDI             254                    4   

                     cols_after_header_promotion  rows_after_decor_drop  \
sheet                                                                     
Table 1. HDI                                  15                    274   
Table 2. HDI trends                           27                    237   
Table 3. IHDI    

In [21]:
import pandas as pd
import re
input_file =r"C:\Users\user\Downloads\HDR25_Statistical_Annex_Tables_1-7(1).xlsx"  # <- original file (no overwrite)
output_file = r"C:\Users\user\Downloads\cleaned_outputs"  # will be created if missing

# patterns that represent segregation/category rows
seg_patterns = [
    r"very high human development",
    r"high human development",
    r"medium human development",
    r"low human development",
]

pattern = re.compile("|".join(seg_patterns), flags=re.IGNORECASE)

xls = pd.ExcelFile(input_file)
writer = pd.ExcelWriter(output_file, engine="openpyxl")

for sheet in xls.sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet)

    # drop fully blank rows
    df = df.dropna(how="all")

    # remove segregation rows ANYWHERE in the sheet
    df = df[~df.apply(lambda row: row.astype(str).str.contains(pattern).any(), axis=1)]

    # now drop rows with ANY missing values
    df = df.dropna(how="any")

    df.to_excel(writer, sheet_name=sheet, index=False)

writer.save()
print("Cleaned rows + removed segregation categories → saved to temp.xlsx")


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\user\\Downloads\\cleaned_outputs'

In [23]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# -------------------------
# Config
# -------------------------
input_file = "HDR25_Statistical_Annex_Tables_1-7.xlsx"  # your input (don't overwrite)
preferred_output = Path("temp.xlsx")                   # primary target
# segmentation patterns to remove (case-insensitive)
seg_patterns = [
    r"very high human development",
    r"high human development",
    r"medium human development",
    r"low human development",
    r"very high", r"high", r"medium", r"low", r"average", r"avg", r"total"
]
seg_regex = re.compile("|".join(seg_patterns), flags=re.IGNORECASE)

# -------------------------
# Cleaning function (rows only)
# -------------------------
def clean_sheet_rows(df):
    # 1) drop rows that are fully blank
    df = df.dropna(how="all")
    if df.shape[0] == 0:
        return df

    # 2) remove segregation rows: any row where any cell matches seg_regex
    # (We keep this conservative: test stringified cell values)
    mask_seg = df.astype(str).apply(lambda row: row.str.contains(seg_regex).any(), axis=1)
    if mask_seg.any():
        df = df[~mask_seg]

    # 3) drop rows with ANY missing value
    df = df.dropna(how="any")

    return df

# -------------------------
# Build cleaned sheets dict
# -------------------------
xls = pd.ExcelFile(input_file)
cleaned_sheets = {}
summary = []

for sheet in xls.sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet, dtype=object)
    rows_before = len(df)
    cols_before = len(df.columns)

    cleaned = clean_sheet_rows(df)

    rows_after = len(cleaned)
    cols_after = len(cleaned.columns)

    cleaned_sheets[sheet] = cleaned
    summary.append({
        "sheet": sheet,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "cols_before": cols_before,
        "cols_after": cols_after
    })
    print(f"[{sheet}] rows {rows_before} -> {rows_after} | cols {cols_before} -> {cols_after}")

# -------------------------
# Safe save: try preferred output, fallback with timestamp on PermissionError
# -------------------------
def safe_save_excel(cleaned_dict, preferred_path: Path) -> Path:
    """Try to save to preferred_path; on PermissionError save to timestamped fallback and return path used."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    try:
        with pd.ExcelWriter(preferred_path, engine="openpyxl") as writer:
            for sheet_name, df in cleaned_dict.items():
                # keep same sheet names (Excel limit 31 chars)
                out_name = str(sheet_name)[:31]
                df.to_excel(writer, sheet_name=out_name, index=False)
        print("Saved cleaned workbook to:", preferred_path)
        return preferred_path
    except PermissionError as e:
        print(f"PermissionError saving to {preferred_path}: {e}")
        fallback = preferred_path.with_name(preferred_path.stem + "_" + ts + preferred_path.suffix)
        try:
            with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
                for sheet_name, df in cleaned_dict.items():
                    out_name = str(sheet_name)[:31]
                    df.to_excel(writer, sheet_name=out_name, index=False)
            print("Saved cleaned workbook to fallback path:", fallback)
            return fallback
        except Exception as e2:
            print("Failed to save fallback file:", e2)
            raise

# attempt save
used_path = safe_save_excel(cleaned_sheets, preferred_output)

# print summary table
print("\nSummary per sheet:")
print(pd.DataFrame(summary).set_index("sheet"))
print("\nOpen this file to inspect results:", used_path)
print("If you see PermissionError again: close any open Excel file named", preferred_output.name, "and re-run.")


FileNotFoundError: [Errno 2] No such file or directory: 'HDR25_Statistical_Annex_Tables_1-7.xlsx'

In [25]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# --------------------------------------------------------
# 1) FIX THIS PATH → PUT YOUR REAL FILE LOCATION HERE
# --------------------------------------------------------
input_file = r"C:\Users\user\Downloads\HDR25_Statistical_Annex_Tables_1-7.xlsx"

preferred_output = Path("temp.xlsx")

# segregation patterns to delete
seg_patterns = [
    r"very high human development",
    r"high human development",
    r"medium human development",
    r"low human development",
    r"very high",
    r"high",
    r"medium",
    r"low",
    r"average",
    r"avg",
    r"total"
]

seg_regex = re.compile("|".join(seg_patterns), flags=re.IGNORECASE)


# --------------------------------------------------------
# Cleaning: remove segregation rows + rows with any NA
# --------------------------------------------------------
def clean_sheet(df):
    # drop completely empty rows first
    df = df.dropna(how="all")

    # remove segregation rows
    mask_seg = df.astype(str).apply(lambda row: row.str.contains(seg_regex).any(), axis=1)
    df = df[~mask_seg]

    # remove rows with *any* empty cell
    df = df.dropna(how="any")

    return df


# --------------------------------------------------------
# Safe save helper
# --------------------------------------------------------
def safe_save(cleaned_dict, preferred_path):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    try:
        with pd.ExcelWriter(preferred_path, engine="openpyxl") as writer:
            for sheet, df in cleaned_dict.items():
                df.to_excel(writer, sheet_name=str(sheet)[:31], index=False)
        print("Saved cleaned workbook to:", preferred_path)
        return preferred_path
    except PermissionError:
        fallback = preferred_path.with_name(preferred_path.stem + "_" + ts + preferred_path.suffix)
        with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
            for sheet, df in cleaned_dict.items():
                df.to_excel(writer, sheet_name=str(sheet)[:31], index=False)
        print("temp.xlsx was locked → saved instead to:", fallback)
        return fallback


# --------------------------------------------------------
# Main execution
# --------------------------------------------------------
xls = pd.ExcelFile(input_file)

cleaned = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet, dtype=object)
    cleaned_df = clean_sheet(df)
    cleaned[sheet] = cleaned_df
    print(f"[{sheet}] rows before={len(df)}, after={len(cl


SyntaxError: unterminated string literal (detected at line 80) (751793495.py, line 80)

In [31]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# --------------------------------------------------------
# 1) SET YOUR INPUT FILE PATH HERE
# --------------------------------------------------------
input_file = r"C:\Users\user\Downloads\HDR25_Statistical_Annex_Tables_1-7.xlsx"

preferred_output = Path(r"C:\Users\user\Downloads\temp.xlsx")

# Segregation keywords pattern
seg_patterns = [
    r"very high human development",
    r"high human development",
    r"medium human development",
    r"low human development",
    r"very high",
    r"high",
    r"medium",
    r"low",
    r"average",
    r"avg",
    r"total"
]

seg_regex = re.compile("|".join(seg_patterns), flags=re.IGNORECASE)


# --------------------------------------------------------
# CLEAN ONE SHEET
# --------------------------------------------------------
def clean_sheet(df):
    # Drop completely empty rows first
    df = df.dropna(how="all")

    # Remove segregation rows (category rows in purple)
    mask_seg = df.astype(str).apply(lambda row: row.str.contains(seg_regex).any(), axis=1)
    df = df[~mask_seg]

    # Remove rows with ANY missing value
    df = df.dropna(how="any")

    return df


# --------------------------------------------------------
# SAFE SAVE — handles PermissionError if temp.xlsx is open
# --------------------------------------------------------
def safe_save(cleaned_dict, preferred_path):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    try:
        with pd.ExcelWriter(preferred_path, engine="openpyxl") as writer:
            for sheet, df in cleaned_dict.items():
                df.to_excel(writer, sheet_name=str(sheet)[:31], index=False)
        print("Saved cleaned workbook to:", preferred_path)
        return preferred_path

    except PermissionError:
        fallback = preferred_path.with_name(preferred_path.stem + "_" + ts + preferred_path.suffix)
        with pd.ExcelWriter(fallback, engine="openpyxl") as writer:
            for sheet, df in cleaned_dict.items():
                df.to_excel(writer, sheet_name=str(sheet)[:31], index=False)
        print("temp.xlsx is open → saved instead to:", fallback)
        return fallback


# --------------------------------------------------------
# MAIN
# --------------------------------------------------------
xls = pd.ExcelFile(input_file)
cleaned = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet, dtype=object)
    cleaned_df = clean_sheet(df)

    cleaned[sheet] = cleaned_df
    print(f"[{sheet}] rows before = {len(df)}, after = {len(cleaned_df)}")

output_used = safe_save(cleaned, preferred_output)

print("\nCleaned file saved at:", output_used)


[Table 1. HDI] rows before = 278, after = 194
[Table 2. HDI trends] rows before = 241, after = 194
[Table 3. IHDI] rows before = 266, after = 194
[Table 4. GDI] rows before = 278, after = 194
[Table 5. GII] rows before = 267, after = 193
[Table 7. PHDI] rows before = 253, after = 194
Saved cleaned workbook to: C:\Users\user\Downloads\temp.xlsx

Cleaned file saved at: C:\Users\user\Downloads\temp.xlsx


In [1]:
import pandas as pd

# Read all sheets into a dictionary
sheets = pd.read_excel(r"C:\Users\user\Downloads\temp.xlsx", sheet_name=None)

cleaned_sheets = {}

for sheet_name, df in sheets.items():
    # Remove rows where ANY cell contains ".."
    df_clean = df[~df.isin([".."]).any(axis=1)]
    cleaned_sheets[sheet_name] = df_clean

# Write all cleaned sheets to a new Excel file
with pd.ExcelWriter("fintemp.xlsx", engine="openpyxl") as writer:
    for sheet_name, df_clean in cleaned_sheets.items():
        df_clean.to_excel(writer, sheet_name=sheet_name, index=False)


In [3]:
import pandas as pd
import os

input_file = r"C:\Users\user\Downloads\temp.xlsx"
output_file = r"C:\Users\user\Downloads\fintemp.xlsx"

sheets = pd.read_excel(input_file, sheet_name=None)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df = df[~df.isin([".."]).any(axis=1)]
        df.to_excel(writer, sheet_name=sheet_name, index=False)
